# Setup

Uses `opik-ai-evaluations/.venv` (works even if Jupyter picks system Python).

One-time terminal setup:

```bash
cd opik-ai-evaluations
python3 -m venv .venv
.venv/bin/pip install -U pip opik pandas openai ipykernel
.venv/bin/python -m ipykernel install --user --name=opik-eval --display-name="Python (opik-eval)"
```

Run the first code cell below before the rest of the notebook.


In [37]:
import sys
from pathlib import Path

_repo = Path.cwd()
if not (_repo / "notebook_env.py").exists():
    _fallback = Path("/home/nxtwave/Repositories/opik-ai-evaluations")
    _repo = _fallback if (_fallback / "notebook_env.py").exists() else _repo
sys.path.insert(0, str(_repo))

from notebook_env import activate

activate()

import opik
import pandas as pd
print("OK —", sys.executable)
print("opik", opik.__version__, "| pandas", pd.__version__)


Note: Jupyter is not using the .venv interpreter (prefix=/usr).
Loaded packages from /home/nxtwave/Repositories/opik-ai-evaluations/.venv/lib/python3.12/site-packages
Loaded environment from /home/nxtwave/Repositories/opik-ai-evaluations/.env
OK — /usr/bin/python
opik 2.2.100 | pandas 3.0.6


In [ ]:
import sys
from pathlib import Path

_repo = Path.cwd()
if not (_repo / "notebook_env.py").exists():
    _fallback = Path("/home/nxtwave/Repositories/opik-ai-evaluations")
    if (_fallback / "notebook_env.py").exists():
        sys.path.insert(0, str(_fallback))

import pandas as pd
import json
import os
import re
from opik import Opik
from opik.evaluation.metrics import base_metric, score_result
from opik.integrations.openai import track_openai
from opik.evaluation import evaluate
from openai import OpenAI
from typing import Any

# Opik Configuration
os.environ["OPIK_URL_OVERRIDE"] = "https://3.110.54.210/api"
os.environ["OPIK_CHECK_TLS_CERTIFICATE"] = "false"
import importlib
import notebook_env
importlib.reload(notebook_env)
notebook_env.load_env_file(override=True)
if not os.environ.get("OPENROUTER_API_KEY"):
    raise ValueError(
        "Set OPENROUTER_API_KEY in opik-ai-evaluations/.env (copy from .env.example)"
    )
os.environ["OPIK_PROJECT_NAME"] = "Video Session z-ai/glm-4.7"

# Experiment Configuration
EXPERIMENT_NAME = "video-session-eval_z-ai/glm-4.7"
MODEL_NAME = "z-ai/glm-4.7"
NB_SAMPLES = 10  # None = evaluate full dataset
DATASET_NAME = "ai_tutor_video_session_prod"

# OpenRouter client
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

# Opik client
opik_client = Opik(project_name=os.environ["OPIK_PROJECT_NAME"])

print("✅ Environment configured")

✅ Environment configured


# Upload Dataset

In [39]:
dataset = opik_client.get_or_create_dataset(DATASET_NAME)
dataset.get_version_info = lambda: None

print("name:", dataset.name)
print("project:", dataset.project_name)
print("id:", dataset.id)
current_size = dataset.dataset_items_count or 0
print("item count:", current_size)

if current_size == 0:
    df = pd.read_csv("your_dataset.csv")
    dataset.insert_from_pandas(dataframe=df)
    print(f"✅ Uploaded {len(df)} rows to '{DATASET_NAME}'")
else:
    print(f"✅ Dataset '{DATASET_NAME}' already exists with {current_size} rows, skipping upload")
    df = dataset.to_pandas()
    print("columns:", df.columns.tolist())
    df.head()

name: ai_tutor_video_session_prod
project: Video Session anthropic/claude-opus-5.5-reasoning
id: 0199ecf5-1bd6-7b6f-9821-2edc5e91ca46
item count: 107
✅ Dataset 'ai_tutor_video_session_prod' already exists with 107 rows, skipping upload
columns: ['ref_output', 'input', 'user_id', 'conversation_id', 'current_prompt', 'bot_id', 'id']


# Set Up Model Config

In [40]:
# ── All metrics use this shared evaluator LLM ──────────────────────────────
EVALUATOR_MODEL = "google/gemini-2.5-pro"
def call_evaluator(prompt: str) -> dict:
    response = openrouter_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.001,
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"```json|```", "", raw).strip()
    return json.loads(raw)

print(f"✅ Evaluator LLM configured: {EVALUATOR_MODEL}")

import importlib

import eval_common
import metrics.video_session as video_session_metrics_mod

importlib.reload(eval_common)
importlib.reload(video_session_metrics_mod)

from eval_common import call_judge, make_tracked_llm_call, parse_input_messages, reset_judge_duration
from metrics.video_session import (
    build_video_session_metrics,
    video_evaluation_task_extras,
)


def _judge(prompt: str, allowed_scores=None):
    return call_judge(openrouter_client, EVALUATOR_MODEL, prompt, allowed_scores)

✅ Evaluator LLM configured: google/gemini-2.5-pro


# Create you custom evaluators

# ClosingStatementDetectionMetric

In [41]:
class ClosingStatementDetectionMetric(base_metric.BaseMetric):
    """
    Scores -1 if student input is NOT a closing statement.
    Scores  1 if it IS a closing statement AND response handles it correctly.
    Scores  0 if it IS a closing statement BUT response fails criteria.
    """
    def __init__(self, name: str = "closing_statement_detection"):
        self.name = name
        self.prompt_template = """
You are evaluating whether the AI tutor correctly detected and responded to a student's closing statement in a video session tutoring context.

A closing statement is when the student expresses satisfaction and wants to conclude the current conversation (e.g., "thank you", "got it", "understood", "thanks", "okay got it").

## Evaluation Criteria:
When the student input IS a closing statement, the response MUST meet ALL of the following:
- Correctly identifies the satisfaction/conclusion signal
- Provides a brief, encouraging closing response (10-15 words max)
- Sets should_student_reply: "No"
- Sets reason_flag: "No"
- Does NOT continue explaining or ask follow-up questions
- Uses varied responses

## Context Provided:
Previous Conversation: {prev_conversation}
Student Input: {student_input}
Tutor Response to Evaluate: {output}

## Scoring:
Score -1 if the student input is NOT a closing statement.
Score  1 if the student input IS a closing statement AND the response correctly handles it by meeting ALL criteria listed above.
Score  0 if the student input IS a closing statement BUT the response fails ANY criteria.

Be extremely strict. Even one violation means the score is 0.

Return your evaluation as a JSON object:
{{
    "reason": "<detailed explanation with specific examples from the response>",
    "score": <-1 or 0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(self, output: str, prev_conversation: str = "", input_messages: list = None, **ignored_kwargs: Any):
        student_input = ""
        if input_messages:
            for msg in reversed(input_messages):
                if isinstance(msg, dict) and msg.get("role") == "user":
                    student_input = msg.get("content", "")
                    break

        prompt = self.prompt_template.format(
            prev_conversation=prev_conversation,
            student_input=student_input,
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ ClosingStatementDetectionMetric defined")

✅ ClosingStatementDetectionMetric defined


# ResponseFormatComplianceMetric

In [42]:
class ResponseFormatComplianceMetric(base_metric.BaseMetric):
    """
    Scores 1 if response strictly follows all JSON formatting requirements.
    Scores 0 if response violates ANY requirement.
    """
    def __init__(self, name: str = "response_format_compliance"):
        self.name = name
        self.prompt_template = """
You are evaluating whether the AI tutor response follows the required JSON format structure and content guidelines for video session tutoring.

## Required Format:
{{
  "reply": "<text only, no code, 40-50 words max>",
  "should_student_reply": "Yes" / "No",
  "reason_flag": "Yes" / "No"
}}

## Evaluation Criteria:
The response MUST meet ALL of the following:
- Valid JSON structure (starts with {{, ends with }})
- Contains exactly three keys: reply, should_student_reply, reason_flag
- No markdown formatting (no triple backticks, no ```json)
- No code blocks or code fences around the JSON
- No extra text or characters outside the JSON object
- The reply field contains NO code snippets or syntax examples
- should_student_reply value is either "Yes" or "No" (not true/false or 1/0)
- reason_flag value is either "Yes" or "No" (not true/false or 1/0)

## Context Provided:
Previous Conversation: {prev_conversation}
Student Input: {student_input}
Tutor Response to Evaluate: {output}

## Scoring:
Score 1 if the response strictly follows ALL formatting requirements.
Score 0 if the response violates ANY requirement.

Be extremely strict. Even a single violation means score is 0.

Return your evaluation as a JSON object:
{{
    "reason": "<detailed explanation with specific examples from the response>",
    "score": <0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(self, output: str, prev_conversation: str = "", input_messages: list = None, **ignored_kwargs: Any):
        student_input = ""
        if input_messages:
            for msg in reversed(input_messages):
                if isinstance(msg, dict) and msg.get("role") == "user":
                    student_input = msg.get("content", "")
                    break

        prompt = self.prompt_template.format(
            prev_conversation=prev_conversation,
            student_input=student_input,
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ ResponseFormatComplianceMetric defined")

✅ ResponseFormatComplianceMetric defined


# ResponseQualityMetric

In [43]:
class ResponseQualityMetric(base_metric.BaseMetric):
    """
    Scores 1 if response meets ALL communication and explanation quality criteria.
    Scores 0 if response fails ANY criterion.
    """
    def __init__(self, name: str = "response_quality"):
        self.name = name
        self.prompt_template = """
You are evaluating the quality of the AI tutor's response in a video session tutoring context, focusing on tone, clarity, and communication effectiveness.

## Evaluation Criteria:
The response MUST meet ALL of the following:

Communication Quality:
- Response length: 40-50 words maximum
- Always responds in English (regardless of student's query language)
- Supportive, encouraging tone (never harsh or dismissive)
- Natural, conversational language (filler words like "Hmm", "I see", "Okay" are acceptable)
- Sounds human-like, not robotic

Explanation Quality:
- Clear conceptual explanations without code snippets
- Uses analogies or real-world examples to explain concepts
- Acknowledges student's understanding level and progress
- No syntax, code examples, or programming instructions
- Focuses on helping student understand "why", not just "what"

## Context Provided:
Previous Conversation: {prev_conversation}
Student Input: {student_input}
Tutor Response to Evaluate: {output}

## Scoring:
Score 1 if the response meets ALL criteria listed above.
Score 0 if the response fails ANY criterion.

Be extremely strict. Even one violation means score is 0.

Return your evaluation as a JSON object:
{{
    "reason": "<detailed explanation with specific examples from the response>",
    "score": <0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(self, output: str, prev_conversation: str = "", input_messages: list = None, **ignored_kwargs: Any):
        student_input = ""
        if input_messages:
            for msg in reversed(input_messages):
                if isinstance(msg, dict) and msg.get("role") == "user":
                    student_input = msg.get("content", "")
                    break

        prompt = self.prompt_template.format(
            prev_conversation=prev_conversation,
            student_input=student_input,
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ ResponseQualityMetric defined")

✅ ResponseQualityMetric defined


# IntentUnderstandingMetric

In [44]:
class IntentUnderstandingMetric(base_metric.BaseMetric):
    """
    Scores 1 if the tutor correctly understood the student's intent.
    Scores 0 if the tutor misunderstood or answered a different question.
    """
    def __init__(self, name: str = "intent_understanding"):
        self.name = name
        self.prompt_template = """
You are evaluating whether the AI tutor correctly understood the student's intended question, even when the question is vague, unclear, or grammatically incorrect.

## Evaluation Criteria:
The response MUST demonstrate ANY of the following:
- Addresses what the student actually meant, not just what they literally said
- Interprets vague or broken sentences appropriately based on context
- Uses previous conversation history to clarify the current query
- Handles grammar mistakes, transcription errors, or incomplete thoughts
- Asks appropriate clarifying questions when intent is genuinely unclear
- Makes reasonable assumptions based on video content and context

## Context Provided:
Previous Conversation: {prev_conversation}
Student Input: {student_input}
Tutor Response to Evaluate: {output}

## Scoring:
Score 1 if the response demonstrates correct understanding of student's intent (meets ANY of the criteria above).
Score 0 if the response misunderstands the intent, answers a different question, or fails to address the core confusion.

Be extremely strict. If the response clearly misunderstands what the student was asking, score is 0.

Return your evaluation as a JSON object:
{{
    "reason": "<detailed explanation with specific examples from the response>",
    "score": <0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(self, output: str, prev_conversation: str = "", input_messages: list = None, **ignored_kwargs: Any):
        student_input = ""
        if input_messages:
            for msg in reversed(input_messages):
                if isinstance(msg, dict) and msg.get("role") == "user":
                    student_input = msg.get("content", "")
                    break

        prompt = self.prompt_template.format(
            prev_conversation=prev_conversation,
            student_input=student_input,
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ IntentUnderstandingMetric defined")

✅ IntentUnderstandingMetric defined


# BoundaryEnforcementMetric

In [45]:
class BoundaryEnforcementMetric(base_metric.BaseMetric):
    """
    Scores -1 if student query is on-topic (metric not applicable).
    Scores  1 if query is off-topic AND response correctly handles it.
    Scores  0 if query is off-topic BUT response fails to handle it correctly.
    """
    def __init__(self, name: str = "boundary_enforcement"):
        self.name = name
        self.prompt_template = """
You are evaluating whether the AI tutor correctly identifies and handles off-topic, inappropriate, or out-of-scope queries in a video session tutoring context.

## Evaluation Criteria:

Off-Topic Detection — response MUST meet ALL relevant criteria when query is off-topic:
- Correctly identifies queries unrelated to Python learning or video session content
- Identifies nonsensical/random inputs (e.g., "blue", "hello", gibberish)
- Politely declines and redirects student back to video content
- Sets reason_flag: "Yes" for off-topic queries
- Does NOT engage with or answer off-topic requests

Assessment/Homework Protection:
- Refuses to provide direct answers to assessment questions
- Refuses to provide direct solutions to homework problems
- Offers only conceptual guidance, not complete solutions
- Redirects to understanding concepts rather than getting answers

## Context Provided:
Previous Conversation: {prev_conversation}
Student Input: {student_input}
Tutor Response to Evaluate: {output}

## Scoring:
Score -1 if the student query is on-topic and appropriate (related to Python learning or video content).
Score  1 if the query is off-topic/inappropriate AND the response correctly handles it by meeting ALL relevant criteria above.
Score  0 if the query is off-topic/inappropriate BUT the response fails to handle it correctly.

Be extremely strict.

Return your evaluation as a JSON object:
{{
    "reason": "<detailed explanation with specific examples from the response>",
    "score": <-1 or 0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(self, output: str, prev_conversation: str = "", input_messages: list = None, **ignored_kwargs: Any):
        student_input = ""
        if input_messages:
            for msg in reversed(input_messages):
                if isinstance(msg, dict) and msg.get("role") == "user":
                    student_input = msg.get("content", "")
                    break

        prompt = self.prompt_template.format(
            prev_conversation=prev_conversation,
            student_input=student_input,
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ BoundaryEnforcementMetric defined")

✅ BoundaryEnforcementMetric defined


# VideoContextHandlingMetric

In [46]:
class VideoContextHandlingMetric(base_metric.BaseMetric):
    def __init__(self, name: str = "video_context_handling"):
        self.name = name
        self.prompt_template = """
You are evaluating whether the AI tutor correctly uses video session content in its response while adhering to timestamp reference rules.

## Evaluation Criteria:

Video Content Relevance:
- Response relates to concepts discussed in the video session
- References or explains topics covered in the video transcript
- Uses the timestamp context internally to locate relevant content
- If topic is covered later in video, mentions: "This will be covered later in the session" or similar

Timestamp Reference Prohibition:
- NEVER mentions "timestamp" in the reply text
- NEVER mentions "current timestamp" in the reply text
- NEVER mentions "at this timestamp" or "at timestamp" in the reply text
- NEVER mentions specific time markers (e.g., "00:31:00", "31 minutes") in the reply text
- Does not reference time-related phrases in the reply

## Context Provided:
Previous Conversation: {prev_conversation}
Student Input: {student_input}
Timestamp Context: {timestamp}
Tutor Response to Evaluate: {output}

## Scoring:
Score 1 if the response meets ALL criteria (relates to video content AND never mentions timestamps in reply)
Score 0 if the response fails ANY criterion (doesn't relate to video content OR mentions timestamps in reply text)

Be extremely strict. Even one timestamp mention means the score is 0.

Return your evaluation as a JSON object:
{{
    "reason": "<detailed explanation with specific examples from the response>",
    "score": <0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(self, output: str, prev_conversation: str = "", input_messages: list = None, timestamp: str = "", **ignored_kwargs: Any):
        student_input = ""
        if input_messages:
            for msg in reversed(input_messages):
                if isinstance(msg, dict) and msg.get("role") == "user":
                    student_input = msg.get("content", "")
                    break

        prompt = self.prompt_template.format(
            prev_conversation=prev_conversation,
            student_input=student_input,
            timestamp=timestamp,
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ VideoContextHandlingMetric defined")

✅ VideoContextHandlingMetric defined


# Assemble Metrics

In [47]:
video_session_metrics = build_video_session_metrics(_judge)

print(f"✅ {len(video_session_metrics)} metrics ready:")
for m in video_session_metrics:
    print(f"   • {m.name}")

✅ 12 metrics ready:
   • closing_statement_detection
   • response_format_compliance
   • reply_word_count
   • response_quality
   • intent_understanding
   • boundary_enforcement
   • reason_flag_consistency
   • video_context_handling
   • transcript_grounding
   • video_session_alignment
   • grader_duration_sec
   • judge_duration_sec


# Experiment — Run Evaluation

In [48]:
dataset = opik_client.get_dataset(name=DATASET_NAME)
tracked_llm_client = track_openai(openrouter_client)

def llm_call(dataset_item: dict) -> str:
    conversation = parse_input_messages(dataset_item.get("input"))
    prompt_template = dataset_item.get("current_prompt", "") or ""
    messages = [{"role": "system", "content": prompt_template}]
    for msg in conversation:
        if isinstance(msg, dict) and msg.get("role") in ("user", "assistant"):
            messages.append({"role": msg["role"], "content": msg["content"]})
    return make_tracked_llm_call(tracked_llm_client, MODEL_NAME, messages)


import time


def evaluation_task(dataset_item: dict) -> dict:
    reset_judge_duration()
    grader_start = time.perf_counter()
    output = llm_call(dataset_item)
    grader_duration_sec = time.perf_counter() - grader_start
    return {
        "output": output,
        **video_evaluation_task_extras(dataset_item),
        "grader_duration_sec": round(grader_duration_sec, 3),
    }

print(f"🚀 Starting evaluation : {EXPERIMENT_NAME}")
print(f"📁 Dataset             : {DATASET_NAME}")
print(f"🤖 Model               : {MODEL_NAME}")
print(f"📊 Metrics             : {len(video_session_metrics)}")
print(f"Evaluate object : {evaluate}")

dataset.get_version_info = lambda: None

eval_results = evaluate(
    experiment_name=EXPERIMENT_NAME,
    dataset=dataset,
    task=evaluation_task,
    scoring_metrics=video_session_metrics,
    task_threads=2,
    verbose=1,
    nb_samples=NB_SAMPLES,
)

print("✅ Evaluation completed")

🚀 Starting evaluation : video-session-eval_anthropic/claude-opus-5.5-low-reasoning
📁 Dataset             : ai_tutor_video_session_prod
🤖 Model               : anthropic/claude-opus-5.5
📊 Metrics             : 12
Evaluate object : <function evaluate at 0x70515a724900>


Evaluation (there might be a delay before the first items are processed):   0%|          | 0/10 [00:00<?, ?it/…

OPIK: Started logging traces to the "Video Session anthropic/claude-opus-5.5-reasoning" project at https://3.110.54.210/api/v1/session/redirect/projects/?trace_id=01a15436-9ba4-7af1-bd1b-8107b218981d&path=aHR0cHM6Ly8zLjExMC41NC4yMTAvYXBp.


╭─ ai_tutor_video_session_prod (10 samples) ───────────╮
│                                                      │
│ Total time:        00:12:08                          │
│ Number of samples: 10                                │
│                                                      │
│ closing_statement_detection: 0.0000 (avg) - 8 failed │
│ response_format_compliance: 0.9000 (avg)             │
│ reply_word_count: 1.0000 (avg) - 1 failed            │
│ response_quality: 0.9000 (avg)                       │
│ intent_understanding: 0.9000 (avg)                   │
│ boundary_enforcement: 0.5000 (avg) - 8 failed        │
│ reason_flag_consistency: 0.7000 (avg)                │
│ video_context_handling: 1.0000 (avg) - 1 failed      │
│ transcript_grounding: 0.5000 (avg)                   │
│ video_session_alignment: 0.7000 (avg)                │
│ grader_duration_sec: 5.9082 (avg)                    │
│ judge_duration_sec: 133.8085 (avg)                   │
│                                                      │
╰──────────────────────────────────────────────────────╯

Uploading results to Opik ...

View the results ]8;id=9870189;https://3.110.54.210/opik/default/experiments/0199ecf5-1bd6-7b6f-9821-2edc5e91ca46/compare?experiments=%5B%2201a15436-9a0e-7f1a-b576-c6188464a04e%22%5D\in your Opik dashboard]8;;\.

✅ Evaluation completed
